In [1]:
import sys
!{sys.executable} -m pip install tensorflow

  Using cached tensorflow-2.21.0-cp313-cp313-win_amd64.whl.metadata (4.5 kB)
  Using cached absl_py-2.4.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached protobuf-7.34.1-cp310-abi3-win_amd64.whl.metadata (595 bytes)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached grpcio-1.80.0-cp313-cp313-win_amd64.whl.metadata (3.9 kB)
  Using cached keras-3.14.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached h5py-3.14.0-cp313-cp313-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.5.4-cp313-cp313-win_amd64.whl.metadata (9.2 kB)
  Using cach

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.34.1 which is incompatible.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

In [2]:
# Load dataset
df = pd.read_csv("../Processed-Data/final_combined_dataset.csv")

X = df["email_text"]
y = df["label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tokenization
max_words = 10000
max_len = 200

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')


In [6]:
# CNN Model
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    GlobalMaxPooling1D(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss = 'binary_crossentropy',
    optimizer = 'adam',
    metrics = ['accuracy']
)

# Train model
history = model.fit(
    X_train_pad,
    y_train,
    epochs = 5,
    batch_size=128,
    validation_split=0.2
)

# Evaluate
loss, accuracy = model.evaluate(X_test_pad, y_test)
print("CNN Accuracy:", accuracy)

C:\Users\Laqua\anaconda3\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 109s 280ms/step - accuracy: 0.9471 - loss: 0.1380 - val_accuracy: 0.9874 - val_loss: 0.0396
Epoch 2/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 108s 289ms/step - accuracy: 0.9929 - loss: 0.0238 - val_accuracy: 0.9856 - val_loss: 0.0446
Epoch 3/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 108s 289ms/step - accuracy: 0.9980 - loss: 0.0081 - val_accuracy: 0.9879 - val_loss: 0.0405
Epoch 4/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 108s 290ms/step - accuracy: 0.9995 - loss: 0.0023 - val_accuracy: 0.9879 - val_loss: 0.0492
Epoch 5/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 107s 287ms/step - accuracy: 0.9997 - loss: 0.0012 - val_accuracy: 0.9880 - val_loss: 0.0537
466/466 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - accuracy: 0.9888 - loss: 0.0481
CNN Accuracy: 0.9887979626655579


In [7]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Generate predictions
y_pred_probs = model.predict(X_test_pad)
y_pred = (y_pred_probs > 0.5).astype("int32").flatten()

# Displaying evaluation metrics
print("===== CNN RESULTS =====")
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

466/466 ━━━━━━━━━━━━━━━━━━━━ 8s 15ms/step
===== CNN RESULTS =====
Accuracy: 0.9887979608264019

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      7033
           1       0.99      0.99      0.99      7875

    accuracy                           0.99     14908
   macro avg       0.99      0.99      0.99     14908
weighted avg       0.99      0.99      0.99     14908

Confusion Matrix:
[[6957   76]
 [  91 7784]]
